<a href="https://colab.research.google.com/github/Osondu-ifunanya/Wildlife-corridor-mapping-using-GIS-and-ML-based-resistance-surfaces/blob/main/Wildlife%20Mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
"""
Wildfire Corridor Mapping using GIS and ML-Based Resistance Surfaces
(Synthetic Spatial Data)

Workflow:
1. Generate synthetic GIS layers (slope, vegetation, moisture, distance to roads)
2. Create wildfire susceptibility score
3. Train ML model to predict resistance surface
4. Compute least-cost wildfire corridor
5. Visualize and export results
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from scipy.sparse.csgraph import dijkstra
from scipy.sparse import csr_matrix

np.random.seed(42)

# ----------------------------------------
# 1. Generate Synthetic GIS Layers
# ----------------------------------------

size = 150

# Terrain slope (degrees)
slope = np.random.uniform(0, 30, (size, size))

# Vegetation density (NDVI proxy)
vegetation = np.random.uniform(0.2, 0.9, (size, size))

# Soil moisture
moisture = np.random.uniform(0.1, 0.7, (size, size))

# Distance to roads (human ignition factor)
distance_roads = np.random.uniform(0, 5, (size, size))

# ----------------------------------------
# 2. Create Synthetic Wildfire Resistance
# ----------------------------------------

# Lower resistance = easier fire spread
resistance = (
    slope * 0.5 +
    vegetation * 20 -
    moisture * 30 -
    distance_roads * 5
)

resistance += np.random.normal(0, 3, (size, size))
resistance = np.clip(resistance, 1, None)

# ----------------------------------------
# 3. Prepare ML Dataset
# ----------------------------------------

features = np.stack([
    slope.flatten(),
    vegetation.flatten(),
    moisture.flatten(),
    distance_roads.flatten()
], axis=1)

target = resistance.flatten()

X_train, X_test, y_train, y_test = train_test_split(
    features, target, test_size=0.3, random_state=42
)

# ----------------------------------------
# 4. Train ML Model
# ----------------------------------------

model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

predicted_resistance = model.predict(features)
resistance_map = predicted_resistance.reshape(size, size)

print("Model trained successfully.")

# ----------------------------------------
# 5. Least-Cost Path (Wildfire Corridor)
# ----------------------------------------

# Build adjacency graph for grid
def build_graph(cost_surface):
    rows, cols = cost_surface.shape
    n = rows * cols
    graph = np.zeros((n, n))

    for i in range(rows):
        for j in range(cols):
            idx = i * cols + j
            neighbors = [(i-1,j),(i+1,j),(i,j-1),(i,j+1)]
            for ni,nj in neighbors:
                if 0 <= ni < rows and 0 <= nj < cols:
                    n_idx = ni * cols + nj
                    graph[idx, n_idx] = cost_surface[ni,nj]
    return csr_matrix(graph)

graph = build_graph(resistance_map)

start = 0
end = size*size - 1

dist_matrix, predecessors = dijkstra(graph, indices=start, return_predecessors=True)

# Reconstruct path
path = []
current = end
while current != start and current != -9999:
    path.append(current)
    current = predecessors[current]

corridor_map = np.zeros((size,size))
for p in path:
    i = p // size
    j = p % size
    corridor_map[i,j] = 1

# ----------------------------------------
# 6. Visualization
# ----------------------------------------

plt.figure(figsize=(12,4))

plt.subplot(1,3,1)
plt.imshow(resistance_map, cmap="hot")
plt.title("Wildfire Resistance Surface")
plt.colorbar()

plt.subplot(1,3,2)
plt.imshow(corridor_map, cmap="Reds")
plt.title("Wildfire Corridor")

plt.subplot(1,3,3)
plt.imshow(vegetation, cmap="Greens")
plt.title("Vegetation Density")

plt.tight_layout()
plt.show()

# ----------------------------------------
# 7. Export Results
# ----------------------------------------

output_df = pd.DataFrame({
    "Slope": features[:,0],
    "Vegetation": features[:,1],
    "Moisture": features[:,2],
    "Distance_Roads": features[:,3],
    "Predicted_Resistance": predicted_resistance
})

output_df.to_excel("wildfire_corridor_results.xlsx", index=False)

print("Results exported to wildfire_corridor_results.xlsx")